# HSLS:09 descriptive statistics

Erik, 2026-11-18

This notebook reports descriptive statistics and missing-data diagnostics for
the cleaned HSLS:09 student file (`hsls_clean`): variable storage types and
missingness rates, survey response counts and analytic weights, univariate
distributions for continuous and categorical variables, missing-value
classifications, joint missingness patterns, covariate differences between
observed and missing cases, pairwise correlations, and variance across
socioeconomic status (SES) imputations. Value labels and NCES reserve codes are
documented in `shared/docs/codebook.md`, and missing-data coding rules are documented
in `shared/docs/missing-data.md`.

All student counts are unweighted record counts in `hsls_clean`. Sample-level
shares use all records in `hsls_clean` as the denominator, whereas univariate
summary statistics use records with non-missing values on each variable.
Univariate continuous and categorical tables report both unweighted statistics
and weighted estimates targeting the population of 9th graders in fall 2009;
missing-data, covariate-balance, correlation, and imputation tables are
unweighted.

Because this notebook reads private data (`inputs.private.*`), rendered output
is written to `outputs/private/notebooks` and reports aggregate statistics only.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

fw.scaffold()
import hsls  # functions/hsls.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_rows", 200)

data_spec = fw.data_read("inputs.public.raw.hsls_data_spec", col_types="c" * 14).fillna("")
clean = fw.data_read("inputs.private.final.hsls_clean", dtype={"student_id": str})
raw = fw.data_read(
    "inputs.private.raw.hsls_student",
    usecols=list(dict.fromkeys(data_spec["nces"])),
    dtype={"STU_ID": str},
)

# Missing-value reasons compare raw and clean values row by row
assert raw["STU_ID"].tolist() == clean["student_id"].tolist()
reasons = hsls.missing_reasons(raw, clean, data_spec)
variables = data_spec.set_index("name")
labels = variables["label"]
f"{len(clean):,} students, {clean.shape[1]} variables"

## Variable overview

The table below lists every column in `hsls_clean`. Variable labels, survey
waves, and measurement types come from `hsls_data_spec`, while `Data type`
reports the storage dtype in `hsls_clean`: `student_id` is stored as a string
and all other variables are numeric, with binary, categorical, and ordinal
variables stored as integer codes mapped to labels in the data spec.
`Distinct observed` and `Observed` count non-missing records only;
`Missing (% of sample)` is the unweighted percentage of all students in
`hsls_clean` with a missing value on that variable.

In [ ]:
overview = pd.DataFrame(
    {
        "Label": labels,
        "Wave": variables["wave"],
        "Type": variables["type"],
        "Data type": clean.dtypes.astype(str),
        "Distinct observed": clean.nunique(),
        "Observed": clean.notna().sum(),
        "Missing (% of sample)": (clean.isna().mean() * 100).round(1),
    }
)
overview

## Survey response and analytic weights

HSLS:09 sampled 23,503 9th graders in fall 2009. A weight above 0 identifies a
student included in weighted estimates under that weight: the base-year student
weight (`w1_weight`) requires a base-year student survey, the parent weight
(`w1_weight_parent`) requires a parent survey, and the longitudinal panel
weight (`w2_weight_panel`) requires both the base-year student survey and the
first follow-up student survey.

The first table reports unweighted student counts and unweighted percentages of
all records in `hsls_clean` with each response indicator equal to 1 and each
analytic weight above 0. The cross-tabulation then gives unweighted student
counts jointly by base-year student survey status (`w1_responded`) and first
follow-up student survey status (`w2_responded`), coding missing values on
`w2_responded` as `Response status unknown`.

In [ ]:
flags = ["w1_responded", "w1_parent_responded", "w2_responded"]
weights = ["w1_weight", "w1_weight_parent", "w2_weight_panel"]
sample = pd.DataFrame(
    {
        "Label": labels[flags + weights].values,
        "Students": [int((clean[name] == 1).sum()) for name in flags]
        + [int((clean[name] > 0).sum()) for name in weights],
    },
    index=[*(f"{name} = 1" for name in flags), *(f"{name} > 0" for name in weights)],
)
sample["Share of sample (%)"] = (sample["Students"] / len(clean) * 100).round(1)
sample

In [ ]:
# Unweighted student counts by base-year and first follow-up student survey status
pd.crosstab(
    clean["w1_responded"].map({0: "No base-year student survey", 1: "Base-year student survey"}),
    clean["w2_responded"].map({0: "No first follow-up student survey", 1: "First follow-up student survey"}).fillna("Response status unknown"),
    margins=True,
    margins_name="Total",
)

## Continuous variable distributions

For each continuous variable in `hsls_data_spec`, unweighted summary statistics
(`Unweighted mean`, `Unweighted SD`, extrema, and percentiles) are computed over
all students with a non-missing value on that variable (`Observed`), where
`Unweighted SD` is the sample standard deviation. `Weighted mean` and
`Weighted SD` are computed over students with both a non-missing value and a
positive analytic weight, using the base-year student weight (`w1_weight`) for
base-year variables and the longitudinal panel weight (`w2_weight_panel`) for
the first follow-up math score (`w2_math`). Weighted SDs are population
standard deviations (dividing by the sum of weights without sample
degrees-of-freedom adjustment).

In [ ]:
continuous = data_spec.loc[data_spec["type"] == "continuous", "name"].tolist()
weight_for = {name: "w2_weight_panel" if variables.at[name, "wave"] == "2" else "w1_weight" for name in continuous}

rows = {}
for name in continuous:
    x = clean[name]
    mean, sd = hsls.weighted_mean_sd(x, clean[weight_for[name]])
    rows[name] = {
        "Observed": int(x.notna().sum()),
        "Unweighted mean": x.mean(),
        "Unweighted SD": x.std(),
        "Min": x.min(),
        "5%": x.quantile(0.05),
        "25%": x.quantile(0.25),
        "Median": x.median(),
        "75%": x.quantile(0.75),
        "95%": x.quantile(0.95),
        "Max": x.max(),
        "Weight": weight_for[name],
        "Weighted mean": mean,
        "Weighted SD": sd,
    }
summary = pd.DataFrame(rows).T
numbers = summary.columns.drop("Weight")
summary[numbers] = summary[numbers].astype(float).round(2)
summary

Base-year math (`w1_math`) and first follow-up math (`w2_math`) are
norm-referenced T-scores scaled by NCES to a weighted mean of 50 and a
standard deviation of 10 in each wave. Socioeconomic status (`w1_ses`) and math
self-efficacy (`w1_math_efficacy`) are standardized continuous composites;
negative values represent substantive scores below the reference mean rather
than NCES reserve codes. The histograms below plot unweighted student counts
over non-missing observations for `w1_math`, `w2_math`, `w1_math_efficacy`, and
`w1_ses`.

In [ ]:
shown = ["w1_math", "w2_math", "w1_math_efficacy", "w1_ses"]
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))
for name, ax in zip(shown, axes.flat):
    x = clean[name].dropna()
    sns.histplot(x, bins=40, ax=ax)
    ax.set_title(f"{labels[name]} (n = {len(x):,})", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("Students (unweighted)")
fig.suptitle("Unweighted distributions of observed continuous analysis variables")
fig.tight_layout()

## Categorical variable frequencies

The frequency table summarizes all binary, categorical, and ordinal variables
in `hsls_data_spec` other than the survey response indicators tabulated above.
For each variable, `Students` and `Unweighted (%)` are computed over all
students with a non-missing value on that variable. `Weighted (%)` is the
category's percentage of total positive weight among students with both a
non-missing value and a positive weight, using the parent weight
(`w1_weight_parent`) for variables from the parent survey (identified by NCES
names beginning with `X1PAR`, `X1P2`, or `X1FAM`) and the base-year student
weight (`w1_weight`) for all remaining variables.

In [ ]:
# Survey response flags are reported in the sample table above
categorical = [
    name
    for name in data_spec.loc[data_spec["type"].isin(["binary", "categorical", "ordinal"]), "name"]
    if name not in flags
]
parent_items = [name for name in categorical if variables.at[name, "nces"].startswith(("X1PAR", "X1P2", "X1FAM"))]


def value_label(name, value):
    row = variables.loc[name]
    if row["derive"] == "applies":
        return {0: "Logical skip in the source variable", 1: "Answered"}[value]
    named = hsls.value_labels(row["labels"])
    if not named and row["type"] == "binary":
        named = {0: "No", 1: "Yes"}
    return named.get(value, "")


tables = []
for name in categorical:
    x = clean[name]
    weight = clean["w1_weight_parent" if name in parent_items else "w1_weight"]
    counts = x.value_counts().sort_index()
    weighted = weight[weight > 0].groupby(x).sum().reindex(counts.index, fill_value=0)
    tables.append(
        pd.DataFrame(
            {
                "Variable": name,
                "Value": counts.index.astype(int),
                "Label": [value_label(name, value) for value in counts.index],
                "Students": counts.values,
                "Unweighted (%)": (counts / counts.sum() * 100).round(1).values,
                "Weighted (%)": (weighted / weighted.sum() * 100).round(1).values,
            }
        )
    )
frequencies = pd.concat(tables).set_index(["Variable", "Value"])
frequencies

## Missing values by reason

Comparing each variable in `hsls_clean` row by row against its raw NCES source
in `hsls_student` classifies every missing value into one of three reasons:

- **Nonresponse**: the survey or item applied to the student, but no response
  was recorded. These values are unobserved within the target subpopulation.
- **Logical skip**: the item did not apply to the student under survey routing
  rules, placing the student outside the subpopulation for which the variable
  is defined.
- **Outside the flag**: the raw file records an answer, but `hsls_clean` sets
  the variable to missing unless an applicability flag equals 1, such as parent
  2's employment where there is no second parent.

The table reports unweighted student counts across all records in `hsls_clean`
for `Observed` and each missing-value reason. For variables with any missing
values, the bar chart plots each reason as an unweighted percentage of all
students in `hsls_clean`, ordered by the nonresponse percentage.

In [ ]:
order = ["Observed", "Nonresponse", "Logical skip", "Outside the flag"]
by_reason = reasons.apply(lambda column: column.value_counts()).T.reindex(columns=order).fillna(0).astype(int)
by_reason.insert(0, "Label", labels)
by_reason

In [ ]:
missing_share = by_reason[order[1:]].div(len(clean)).mul(100)
missing_share = missing_share[missing_share.sum(axis=1) > 0].sort_values("Nonresponse")
ax = missing_share.plot.barh(stacked=True, figsize=(8, 7), width=0.8)
ax.set_xlabel("Share of sample (%, unweighted)")
ax.set_ylabel("")
ax.set_title("Missing values by reason among variables with missing data")
ax.legend(title="Reason", loc="lower right")
plt.tight_layout()

## Joint missingness patterns

Joint missingness is tabulated across the core analysis variables: the two math
scores (`w1_math`, `w2_math`), SES (`w1_ses`), math self-efficacy
(`w1_math_efficacy`), sex (`w1_female`), race/ethnicity (`w1_race`), parents'
education (`w1_parents_edu`), and family income (`w1_family_income`). Each row
represents a combination of observed (`·`) and missing (`X`) indicators,
pooling all missing-value reasons. `Students` is the unweighted count and
`Share of sample (%)` is the unweighted percentage of all records in
`hsls_clean`.

In [ ]:
analysis = [
    "w1_math",
    "w2_math",
    "w1_ses",
    "w1_math_efficacy",
    "w1_female",
    "w1_race",
    "w1_parents_edu",
    "w1_family_income",
]
missing = clean[analysis].isna()
patterns = (
    missing.replace({True: "X", False: "·"})
    .value_counts()
    .rename("Students")
    .reset_index()
)
patterns["Share of sample (%)"] = (patterns["Students"] / len(clean) * 100).round(1)
patterns["Missing variables"] = patterns[analysis].eq("X").sum(axis=1)
patterns.head(15)

In [ ]:
complete = int((~missing.any(axis=1)).sum())
print(f"{complete:,} students ({complete / len(clean):.1%}, unweighted) have observed values on all {len(analysis)} analysis variables.")
print(f"There are {len(patterns)} distinct missingness patterns; the 15 most frequent patterns cover {patterns['Students'].head(15).sum() / len(clean):.1%} of all students.")

Missing values cluster by survey instrument: a student without a base-year
student survey misses the base-year math score, SES, and math self-efficacy
together, and nonresponse on the parent survey leaves parents' education and
family income missing.

The table below partitions `hsls_clean` by each survey response indicator
(`w1_responded`, `w1_parent_responded`, `w2_responded`) and reports the
unweighted percentage of students in each response group (`No`, `Yes`, and
`Unknown` where the indicator itself is missing) who are missing each analysis
variable. The heatmap then shows, among students missing the row variable, the
unweighted percentage who are also missing the column variable.

In [ ]:
def share_missing(flag):
    groups = clean[flag].map({0: "No", 1: "Yes"}).fillna("Unknown")
    table = missing.groupby(groups).mean().mul(100).round(1).T
    table.columns = [f"{flag} = {group}" for group in table.columns]
    return table


pd.concat([share_missing(flag) for flag in flags], axis=1)

In [ ]:
# Unweighted conditional percentage missing the column variable given the row variable is missing
both = missing.T.astype(int) @ missing.astype(int)
conditional = both.div(missing.sum(), axis=0).mul(100)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(conditional, annot=True, fmt=".0f", cmap="viridis", vmin=0, vmax=100, cbar_kws={"label": "Conditional missing (%)"}, ax=ax)
ax.set_xlabel("Column variable also missing")
ax.set_ylabel("Row variable missing")
ax.set_title("Unweighted share missing column variable among students missing row variable")
plt.tight_layout()

## Covariate differences between observed and missing cases

When students with missing data differ systematically on observed variables,
complete-case estimation does not represent the full sample. The tables below
partition students by whether the first follow-up math score (`w2_math`) or
parents' education (`w1_parents_edu`) is observed versus missing, and compare
both groups on selected base-year covariates. Within each group, unweighted
means and unweighted sample variances are computed over that group's
non-missing values on the covariate (`n (missing group, covariate observed)`
gives the non-missing covariate count in the group missing the target
variable). The standardized difference equals the unweighted mean in the
missing group minus the unweighted mean in the observed group, divided by the
unweighted pooled sample standard deviation,
`sqrt((var_observed + var_missing) / 2)`. Absolute standardized differences
exceeding roughly 0.1 indicate substantive covariate imbalance.

The density plot displays unweighted kernel density estimates of observed
base-year math scores (`w1_math`) separately for students with observed and
missing first follow-up math scores (`w2_math`), normalized independently
within each group.

In [ ]:
def compare(missing_on, covariates):
    is_missing = clean[missing_on].isna()
    rows = {}
    for name in covariates:
        observed = clean.loc[~is_missing, name].dropna()
        absent = clean.loc[is_missing, name].dropna()
        pooled = np.sqrt((observed.var() + absent.var()) / 2)
        rows[name] = {
            f"Mean ({missing_on} observed)": observed.mean(),
            f"Mean ({missing_on} missing)": absent.mean(),
            "Standardized difference": (absent.mean() - observed.mean()) / pooled,
            "n (missing group, covariate observed)": len(absent),
        }
    return pd.DataFrame(rows).T.round(2)


display(compare("w2_math", ["w1_math", "w1_ses", "w1_math_efficacy", "w1_female", "w1_parents_edu"]))
display(compare("w1_parents_edu", ["w1_math", "w1_math_efficacy", "w1_female"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
groups = clean["w2_math"].isna().map({False: "First follow-up math observed", True: "First follow-up math missing"})
sns.kdeplot(x=clean["w1_math"], hue=groups, common_norm=False, ax=ax)
ax.set_xlabel(labels["w1_math"])
ax.set_ylabel("Density")
ax.set_title("Unweighted base-year math density by first follow-up math availability")
ax.get_legend().set_title("")
plt.tight_layout()

## Pairwise associations among analysis variables

The correlation matrix reports unweighted Pearson correlation coefficients
across the continuous and ordinal analysis variables (`w1_math`, `w2_math`,
`w1_math_efficacy`, `w1_ses`, `w1_parents_edu`, and `w1_family_income`). Each
entry is computed via pairwise deletion over all students with non-missing
values on both variables in the pair, so sample composition varies across
cells; the accompanying count matrix gives the unweighted pairwise non-missing
student counts (with univariate non-missing counts on the diagonal). The
hexagonal binning plot displays unweighted student counts across the joint
distribution of base-year and first follow-up math scores among students with
both scores observed.

In [ ]:
related = ["w1_math", "w2_math", "w1_math_efficacy", "w1_ses", "w1_parents_edu", "w1_family_income"]
display(clean[related].corr().round(2))
observed = clean[related].notna().astype(int)
observed.T @ observed

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
pair = clean[["w1_math", "w2_math"]].dropna()
ax.hexbin(pair["w1_math"], pair["w2_math"], gridsize=45, cmap="viridis", mincnt=1)
ax.set_xlabel(labels["w1_math"])
ax.set_ylabel(labels["w2_math"])
ax.set_title(f"Base-year and first follow-up math scores (n = {len(pair):,}, unweighted)")
plt.tight_layout()

## Socioeconomic status imputations

For a student with nonresponse on the parent survey, `w1_ses` is the arithmetic
mean of five NCES imputations (`w1_ses_imp1` through `w1_ses_imp5`). Averaging
the five draws eliminates within-student imputation variance and attenuates
between-student dispersion relative to any single draw. Grouped by `w1_ses_imputed`
over students with non-missing SES records, the table reports unweighted
student counts, the student-level sample SD across the five imputations
averaged over students in the group, the maximum absolute discrepancy between
`w1_ses` and the five-imputation mean, and the unweighted between-student
sample SDs of `w1_ses` and of the first imputation (`w1_ses_imp1`).

In [ ]:
imputations = [f"w1_ses_imp{i}" for i in range(1, 6)]
status = clean["w1_ses_imputed"].map(
    {0: "0: All components reported", 1: "1: Parent-survey nonresponse", 2: "2: Partial component imputation"}
)
pd.DataFrame(
    {
        "Students": status.value_counts(),
        "Mean within-student imputation SD": clean[imputations].std(axis=1).groupby(status).mean(),
        "Max |w1_ses - imputation mean|": (clean["w1_ses"] - clean[imputations].mean(axis=1)).abs().groupby(status).max(),
        "Between-student SD of w1_ses": clean["w1_ses"].groupby(status).std(),
        "Between-student SD of imputation 1": clean["w1_ses_imp1"].groupby(status).std(),
    }
).round(3)

## Summary of findings

- **Sample weighting.** Weighted by the base-year student weight (`w1_weight`),
  the base-year math score (`w1_math`) has a mean of 50 and a population SD of
  10 over observed respondents, matching the NCES T-score scaling. Its
  unweighted respondent mean is about 51, indicating that unweighted
  respondents overrepresent higher-scoring students relative to the weighted
  target population.
- **Complete-case coverage.** Across `hsls_clean`, 56.2% of students
  (unweighted) have non-missing values on all eight core analysis variables.
  Most incomplete records reflect unit nonresponse to an entire survey: the
  base-year student survey, the parent survey, or the first follow-up student
  survey.
- **Covariate imbalance under missingness.** In unweighted comparisons over
  each group's non-missing base-year observations, students missing the first
  follow-up math score (`w2_math`) scored 0.44 pooled SDs lower on base-year
  math (`w1_math`) and 0.29 pooled SDs lower on SES (`w1_ses`) than students
  with an observed first follow-up math score. Students missing parents'
  education (`w1_parents_edu`), nearly all due to parent-survey nonresponse,
  also scored 0.44 pooled SDs lower on base-year math than students with
  observed parents' education. Because complete-case restriction
  disproportionately excludes lower-scoring and lower-SES students, subsequent
  models must account for missing data rather than rely on listwise deletion.
- **SES imputation variance.** Among students with parent-survey nonresponse,
  `w1_ses` has an unweighted between-student sample SD of 0.54, compared with
  0.82 for a single imputation. Across the five imputations, the average
  within-student sample SD is 0.66. Treating `w1_ses` as an observed point
  value ignores that imputation uncertainty and understates SES dispersion.